# 01a. Logistic Regression: the Model and the Loss

**Goals for this notebook**

- Understand a new kind of question: not *"which class?"* but *"how likely is class 1?"*
- Build the **hypothesis class** of logistic regression: a linear model $w^T x + b$, squashed into
  $[0,1]$ by the **logistic function**.
- Choose a **loss function** for it, and see (with gradient descent, which you already know) why the
  obvious choice, squared error, is a bad one, and **log-loss** is a good one.
- Derive the **gradient** of the log-loss: the formula we'll use to implement logistic regression
  ourselves later.

**How to use this notebook:** read, run every cell in order, and do the exercises when you reach
them. Exercise cells say `# YOUR CODE HERE`; the cell after an exercise usually **checks** your
answer (✅ / ❌). Questions marked **Try it yourself, before reading on** are for thinking; answer
them in your head (or on paper) before you scroll down.

## 0. Setup

Run the next two cells first. The first imports the libraries we use. The second defines a small
`check` helper for the exercises; you don't need to read it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Interactive sliders (optional): if ipywidgets is missing, we show a few fixed pictures instead.
try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed: the interactive cells will show fixed pictures instead.")

In [ ]:
# --- exercise checker: just run this cell -----------------------------------------------
import base64 as _b64, json as _json
_ANS = _json.loads(_b64.b64decode("eyJXMSI6IC0wLjUsICJXMiI6IC0xLCAiVzMiOiAxLjA4LCAiMS4yIjogWy0wLjIwOTQwNTA0NTg2MDI3OTgsIDEuMzcwMDA4MjM3NjM3Mzk2NV0sICIyLjEiOiBbMSwgMCwgMCwgMSwgMSwgMCwgMV0sICIyLjIiOiBbMC41NzE0Mjg1NzE0Mjg1NzE0LCAwLjU3MTQyODU3MTQyODU3MTRdLCAiMy5BIjogWzEuMCwgMC43MzEwNTg1Nzg2MzAwMDQ5LCAtMi4wLCAwLjExOTIwMjkyMjAyMjExNzU1XSwgIjMuQiI6IHsiYXJncyI6IFtbWy01LjAsIC0xLjAsIDAuMCwgMS4wLCA1LjBdXV0sICJvdXQiOiBbWzAuMDA2NjkyODUwOTI0Mjg0ODU1NCwgMC4yNjg5NDE0MjEzNjk5OTUxLCAwLjUsIDAuNzMxMDU4NTc4NjMwMDA0OSwgMC45OTMzMDcxNDkwNzU3MTUzXV19LCAiMy5CMiI6IDAuMCwgIjMuQyI6IFswLjA0NTE3NjY1OTczMDkxMjE0LCAwLjEwNDk5MzU4NTQwMzUwNjUsIDAuMTk2NjExOTMzMjQxNDgxODUsIDAuMjUsIDAuMTk2NjExOTMzMjQxNDgxODUsIDAuMTA0OTkzNTg1NDAzNTA2NjIsIDAuMDQ1MTc2NjU5NzMwOTEyXSwgIjMuRCI6IFsxLjM4NjI5NDM2MTExOTg5MDYsIDYuODMxMDIwNDgxMTEzNTE3XSwgIjMuRSI6IFswLjczMjQwODE5MjQ0NTQwNjYsIC0zLjY2MjA0MDk2MjIyNzAzMjVdLCAiNC5BIjogWzAuNTEwODI1NjIzNzY1OTkwNywgMC4xODAwMDAwMDAwMDAwMDAwMl0sICI0LkIiOiB7ImFyZ3MiOiBbW1sxLCAwLCAxLCAwLCAxXSwgWzAuOSwgMC4yLCAwLjMsIDAuNSwgMC45OV1dXSwgIm91dCI6IFtbMC40NDcxMzQ4Nzc1NDIyODM3NiwgMC4xNTgwMl1dfSwgIjQuQyI6IFsxLjI0NDkxODY2MjQwMzcwOTIsIDAuNjIyNDU5MzMxMjAxODU0NiwgMC42MjI0NTkzMzEyMDE4NTQ2XSwgIjQuRCI6IFstMC45OTkwODg5NDg4MDU1OTk0LCAtMC4wMDE4MTg3ODM4NDQwNTcwMTU3XSwgIjUuQiI6IFswLjQxMDg5MDAwODE4NDA3NDMsIDYuMzAyMDYyNzc5Mzc4ODQ0XX0="))

def _close(a, b, tol):
    try:
        return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol, rtol=0)
    except Exception:
        return False

def check(key, value, tol=1e-3):
    # Compare your answer for exercise `key` with the expected one.
    if value is None or value is Ellipsis or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, _ANS[key], tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_fn(key, fn, tol=1e-3):
    # Call your function on some test inputs and compare with the expected outputs.
    spec = _ANS[key]
    try:
        outs = [fn(*[np.array(a, dtype=float) for a in args]) for args in spec["args"]]
    except Exception as e:
        print(f"❌ {key}: your function raised an error: {type(e).__name__}: {e}")
        return
    if any(o is None for o in outs):
        print(f"⏳ {key}: not answered yet (your function returns None)")
    elif all(_close(o, ex, tol) for o, ex in zip(outs, spec["out"])):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. Your function gives {outs}; check your work and try again")

## Warm-up: what we already know

Three quick questions on the previous unit. Do them by hand first, then type the numbers in.

**W1.** A linear-regression model has $w = (2, -1)$ and $b = 0.5$. What does it predict for
$x = (1, 3)$? (Remember: $h(x) = w^T x + b$.)

**W2.** A perceptron with the same $w$ and $b$ predicts $\text{sign}(w^T x + b)$ with labels
$\{-1,+1\}$. What does it predict for the same $x$?

**W3.** We minimize $f(w) = (w - 3)^2$ with gradient descent, starting from $w = 0$ with learning rate
$\eta = 0.1$. What is $w$ after **two** steps? (Remember: $w \leftarrow w - \eta f'(w)$.)

In [ ]:
# YOUR CODE HERE
w1_answer = None
w2_answer = None
w3_answer = None

check("W1", w1_answer)
check("W2", w2_answer)
check("W3", w3_answer)

## 1. A new kind of question

So far we met two kinds of supervised-learning questions:

- **Regression**: the target is a **number**. *"What will this apartment sell for?"* (linear
  regression)
- **Classification**: the target is a **class**. *"Is this email spam, yes or no?"* (the perceptron)

Very often, though, what we *really* want is a third thing:

- **Probability estimation**: *"How **likely** is class 1?"* The answer is a number in $[0,1]$.

Here are some examples from different fields. In each one the target function takes an example
$x \in \mathbb{R}^n$ and returns the probability that $y = 1$, which we write $P(y = 1 \mid x)$ ("the
probability that $y=1$, given $x$").

| Field | $x$ (the features) | $n$ | $y = 1$ means... | The data |
|---|---|---|---|---|
| Medicine | age, blood pressure, glucose level, BMI, ... | ~8 | the patient develops diabetes within 5 years | medical records of past patients |
| Finance | income, loan amount, years at job, past late payments | ~10 | the borrower defaults on the loan | the bank's history of past loans |
| Email | how often each of 1000 words appears | 1000 | the email is spam | emails that users marked as spam / not spam |
| Sports | team rankings, home/away, recent results | ~6 | the home team wins | results of past games |
| Weather | today's pressure, humidity, temperature, wind | ~5 | it rains tomorrow | years of daily weather records |
| Astronomy | brightness measurements of a star over time | ~50 | the star has a planet | stars already confirmed by telescopes |

In every row the *data* is the same shape: many examples $x_1, \dots, x_N$, each with a label $y_i$
that is either 0 or 1. **We never observe the probability itself**, only what actually happened.
Still, we want the model to output a probability.

**Why a probability, and not just a class?**

- *"It will rain tomorrow"* is much less useful than *"70% chance of rain"*. A 51% forecast and a 99%
  forecast should lead to different decisions (umbrella? cancel the trip?).
- A doctor treats a patient with a 3% risk differently from one with a 45% risk, even though both are
  "probably healthy".
- From a probability we can always get a class (e.g. "predict 1 if the probability is above 0.5"; we
  do this in 01b). We can't go the other way: a class alone doesn't tell us how sure the model is.

### A running example: passing an exam

Throughout this notebook we'll use a small, made-up dataset: for $N = 40$ students we know how many
hours they studied ($x$, so $n = 1$) and whether they **passed** the exam ($y = 1$) or not ($y = 0$).

In [ ]:
rng = np.random.default_rng(1)
N = 40
hours = np.round(rng.uniform(0, 10, N), 1)                        # x: hours studied
true_prob = 1 / (1 + np.exp(-1.2 * (hours - 5)))                  # (hidden from the learner!)
passed = (rng.uniform(size=N) < true_prob).astype(int)            # y: 1 = passed, 0 = failed

print("hours: ", hours[:10], "...")
print("passed:", passed[:10], "...")
print("fraction who passed:", passed.mean())

In [ ]:
plt.scatter(hours, passed, c=passed, cmap="coolwarm", edgecolor="k", zorder=3)
plt.xlabel("hours studied (x)")
plt.ylabel("passed (y)")
plt.yticks([0, 1])
plt.title("Exam data: every label is 0 or 1")
plt.show()

Notice the overlap in the middle: some students who studied 5 hours passed and some didn't. No rule
of the form *"pass if hours > something"* is always right. A **probability** is the natural way to
describe this: *"with 5 hours of study, about half pass; with 9 hours, almost everyone does."*

**Try it yourself, before reading on:** we already know two models. Could we use them here?

- The **perceptron** outputs only $\pm 1$: a class, not a probability.
- **Linear regression** outputs a number. Could we fit a line to the 0/1 labels and read it as a
  probability?

Let's try the second idea.

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(hours.reshape(-1, 1), passed)     # sklearn wants X as a 2D array: one column per feature
slope = lin_reg.coef_[0]
intercept = lin_reg.intercept_
print(f"linear regression: prediction = {slope:.3f} * hours {intercept:+.3f}")

xs = np.linspace(-2, 12, 200)
plt.scatter(hours, passed, c=passed, cmap="coolwarm", edgecolor="k", zorder=3)
plt.plot(xs, slope * xs + intercept, "g-", linewidth=2, label="linear regression line")
plt.axhspan(0, 1, color="gray", alpha=0.1, label="allowed range for a probability")
plt.xlabel("hours studied (x)")
plt.ylabel("y  /  prediction")
plt.legend(loc="upper left")
plt.title("A straight line doesn't stay inside [0, 1]")
plt.show()

The line does go up as hours go up, which is right. But it keeps going: for a student who studies
a lot it predicts a value **above 1**, and for one who doesn't study at all it predicts a value **below
0**. Neither can be a probability. We need a model whose output **always stays between 0 and 1**.
That's what logistic regression is.

### Exercise 1.1: what kind of question?

For each question below, say whether it's **regression**, **classification** or **probability
estimation**. Then say what $x$ could be, roughly what $n$ is, and what $y$ is.

1. What will the highest temperature be in Tel Aviv tomorrow?
2. What is the chance that a customer cancels their phone subscription this month?
3. Which digit (0 to 9) is written in this 28×28 pixel image?

**Your answer:** *(write here)*

### Exercise 1.2: how bad is the line?

Use `slope` and `intercept` from the linear regression above to compute what the line predicts for a
student who studied **0 hours** and for one who studied **12 hours**. Store them in `pred_0` and
`pred_12`.

In [ ]:
# YOUR CODE HERE
pred_0 = None
pred_12 = None
print("prediction for 0 hours: ", pred_0)
print("prediction for 12 hours:", pred_12)
check("1.2", [pred_0, pred_12])

Both are outside $[0,1]$: exactly the problem we saw in the plot.

## 2. A change of labels: from $\{-1,+1\}$ to $\{0,1\}$

With the perceptron we used the labels $y \in \{-1, +1\}$, because the perceptron predicts
$\text{sign}(w^T x + b)$ and the sign function returns $-1$ or $+1$.

**From now on we use $y \in \{0, 1\}$.** Here's why.

- Our model will output **the probability that $y = 1$**. A label of 1 then means *"it happened,
  probability 1"*, and a label of 0 means *"it didn't happen, probability 0"*. The labels live on the
  same scale as the predictions, so it makes sense to compare them directly (for example, predicted
  0.9 vs. actual 1).
- The **average of the labels** is simply the fraction of examples in class 1. We already used this
  above: `passed.mean()` was the fraction of students who passed.
- The loss function we'll build in section 4 has a particularly clean formula with 0/1 labels.

| | Perceptron | Logistic regression |
|---|---|---|
| labels | $y \in \{-1, +1\}$ | $y \in \{0, 1\}$ |
| model output | $\text{sign}(w^T x + b) \in \{-1, +1\}$ | a probability in $[0, 1]$ |
| "the positive class" | $y = +1$ | $y = 1$ |

Converting is easy: $y_{01} = \dfrac{y_{\pm1} + 1}{2}$ turns $-1 \to 0$ and $+1 \to 1$.

(One more convention: we keep the **intercept** $b$ separate from $w$ and write $w^T x + b$. If you
learned the trick of adding a constant feature $x_0 = 1$ so that $b$ becomes part of $w$, that's
exactly the same model, just written differently.)

### Exercise 2.1

Convert `y_pm` (perceptron-style labels) into 0/1 labels and store them in `y01`. Use one line of NumPy
arithmetic, no loops.

In [ ]:
y_pm = np.array([1, -1, -1, 1, 1, -1, 1])
# YOUR CODE HERE
y01 = None
print(y01)
check("2.1", y01)

### Exercise 2.2 (a small twist)

Compute the fraction of positive examples in two ways:

- `frac_a`: from the 0/1 labels `y01` (easy);
- `frac_b`: directly from `y_pm`, **without** converting it first. Hint: what is the mean of `y_pm`,
  and how is it related to the fraction?

In [ ]:
# YOUR CODE HERE
frac_a = None
frac_b = None
print(frac_a, frac_b)
check("2.2", [frac_a, frac_b])

With 0/1 labels this is just `.mean()`. That's one small reason 0/1 labels are convenient.

## 3. The hypothesis class: squash the line into $[0,1]$

Here's the idea. Keep the linear model we know, $z = w^T x + b$ (any real number), and pass it
through a function that **squashes** every real number into the interval $(0,1)$:

$$h(x) = \sigma(w^T x + b)$$

The squashing function we use is the **logistic function** (also called *the sigmoid*):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

The **hypothesis class** of logistic regression is the set of all such functions $h$, one for every
choice of the parameters $w \in \mathbb{R}^n$ and $b \in \mathbb{R}$. Learning means choosing a good
$w$ and $b$, exactly as in linear regression.

### 3.1 Meet the logistic function

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-10, -5, -2, -1, 0, 1, 2, 5, 10]:
    print(f"z = {z:>4}   sigma(z) = {sigmoid(z):.5f}")

In [ ]:
zs = np.linspace(-8, 8, 400)
plt.plot(zs, sigmoid(zs), linewidth=2.5)
plt.axhline(0, color="gray", linewidth=0.8)
plt.axhline(1, color="gray", linewidth=0.8)
plt.axhline(0.5, color="gray", linestyle=":")
plt.axvline(0, color="gray", linestyle=":")
plt.xlabel("z")
plt.ylabel("sigma(z)")
plt.title("The logistic function")
plt.show()

### 3.2 Its properties

Each property below is followed by a short explanation. You'll check several of them yourself in the
exercises.

1. **Range:** $0 < \sigma(z) < 1$ for every $z$. The term $e^{-z}$ is always positive, so the
   denominator $1 + e^{-z}$ is always greater than 1.
2. **The middle:** $\sigma(0) = \frac{1}{1+1} = \frac12$. The model says "50/50" exactly where
   $w^T x + b = 0$.
3. **The ends:** as $z \to +\infty$, $e^{-z} \to 0$ and $\sigma(z) \to 1$. As $z \to -\infty$,
   $e^{-z} \to \infty$ and $\sigma(z) \to 0$.
4. **Monotone:** a larger $z$ always gives a larger $\sigma(z)$. A larger score always means
   "more likely class 1".
5. **Symmetry:** $\sigma(-z) = 1 - \sigma(z)$. The graph looks the same after rotating it
   180° around the point $(0, \tfrac12)$.
6. **A neat derivative:** $\sigma'(z) = \sigma(z)\,\big(1 - \sigma(z)\big)$.

   *Derivation.* Write $\sigma(z) = (1 + e^{-z})^{-1}$. By the chain rule:
   $$\sigma'(z) = -(1+e^{-z})^{-2} \cdot (-e^{-z}) = \frac{e^{-z}}{(1+e^{-z})^2}
   = \underbrace{\frac{1}{1+e^{-z}}}_{\sigma(z)} \cdot \underbrace{\frac{e^{-z}}{1+e^{-z}}}_{1-\sigma(z)}$$
   (For the second bracket: $1 - \sigma(z) = \frac{1 + e^{-z} - 1}{1+e^{-z}} = \frac{e^{-z}}{1+e^{-z}}$.)
   We'll need this in section 4.
7. **The inverse: log-odds.** If $p = \sigma(z)$, then solving for $z$ gives
   $$z = \log\frac{p}{1-p}$$
   The ratio $\frac{p}{1-p}$ is called the **odds** (odds of 3 means "3 times more likely to happen
   than not"), and $\log\frac{p}{1-p}$ is the **log-odds**. So for logistic regression:
   $$\log\frac{h(x)}{1 - h(x)} = w^T x + b$$
   **The model is linear in the log-odds.** Increasing a feature $x_j$ by 1 adds $w_j$ to the
   log-odds.

### 3.3 Zooming in and out

Here's a surprising fact: depending on how far you zoom out, the logistic function looks like three
completely different functions. Move the slider, which sets the plotted range $z \in [-a, a]$.

In [ ]:
def plot_sigmoid_zoom(a=6.0):
    zs = np.linspace(-a, a, 600)
    plt.figure(figsize=(6, 3.5))
    plt.plot(zs, sigmoid(zs), linewidth=2.5)
    plt.ylim(-0.05, 1.05)
    plt.axhline(0.5, color="gray", linestyle=":")
    plt.title(f"sigma(z) for z in [-{a:g}, {a:g}]")
    plt.xlabel("z")
    plt.show()

if HAVE_WIDGETS:
    interact(plot_sigmoid_zoom, a=FloatLogSlider(value=6, base=10, min=-1, max=1.7, step=0.05,
                                                  description="range a"))
else:
    fig, axes = plt.subplots(1, 4, figsize=(15, 3))
    for ax, a in zip(axes, [0.5, 3, 10, 50]):
        zs = np.linspace(-a, a, 600)
        ax.plot(zs, sigmoid(zs), linewidth=2.5)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f"z in [-{a:g}, {a:g}]")
    plt.show()

- **Zoomed in** (small $a$): it looks like a **straight line** through $(0, \tfrac12)$, with slope
  $\sigma'(0) = \tfrac14$. Near the middle the model behaves almost like linear regression.
- **Medium** range: the famous **S-shape**.
- **Zoomed out** (large $a$): it looks like a **step**, jumping from 0 to 1 at $z = 0$. That step is
  the perceptron! (Up to the labels: the perceptron's step goes from $-1$ to $+1$.)

So logistic regression sits between the two models we already know: smooth like linear regression
in the middle, step-like like the perceptron far from the boundary.

### 3.4 The hypothesis on the exam data: what $w$ and $b$ do

With one feature, $h(x) = \sigma(w x + b)$. Use the sliders to find values of $w$ and $b$ that fit
the exam data well. While you do, notice:

- $h(x) = 0.5$ exactly where $w x + b = 0$, that is at $x = -b/w$. This is the **decision
  boundary**.
- **$|w|$ controls steepness**: a larger $|w|$ gives a sharper transition, closer to a step.
- **The sign of $w$ sets the direction**: $w > 0$ means more hours give a higher chance of passing.

In [ ]:
def plot_hypothesis(w=1.0, b=-5.0):
    xs = np.linspace(0, 10, 300)
    plt.figure(figsize=(6, 3.5))
    plt.scatter(hours, passed, c=passed, cmap="coolwarm", edgecolor="k", zorder=3)
    plt.plot(xs, sigmoid(w * xs + b), "g-", linewidth=2.5, label=f"h(x) = sigma({w:.2f} x + {b:.2f})")
    if w != 0 and 0 <= -b / w <= 10:
        plt.axvline(-b / w, color="gray", linestyle="--", label=f"boundary x = {-b / w:.2f}")
    plt.ylim(-0.1, 1.1)
    plt.xlabel("hours studied")
    plt.ylabel("P(pass)")
    plt.legend(loc="upper left", fontsize=9)
    plt.show()

if HAVE_WIDGETS:
    interact(plot_hypothesis, w=FloatSlider(value=1, min=-3, max=5, step=0.1),
             b=FloatSlider(value=-5, min=-25, max=10, step=0.5))
else:
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
    xs = np.linspace(0, 10, 300)
    for ax, (w, b) in zip(axes, [(0.5, -2.5), (1.2, -6), (4, -20), (-1.2, 6)]):
        ax.scatter(hours, passed, c=passed, cmap="coolwarm", edgecolor="k", zorder=3)
        ax.plot(xs, sigmoid(w * xs + b), "g-", linewidth=2.5)
        ax.set_title(f"w = {w}, b = {b}")
        ax.set_xlabel("hours studied")
    plt.show()

Compare this with the line in section 1: every curve here stays inside $(0,1)$, however far you
extend it.

### 3.5 Other "sigmoids"

The logistic function isn't the only way to squash $\mathbb{R}$ into $(0,1)$. Any function that is
increasing, goes from 0 to 1 and passes through $(0, \frac12)$ would do. Three well-known
alternatives:

- **Probit:** $\Phi(z)$, the cumulative distribution function of the standard normal distribution
  (the "bell curve"). $\Phi(z)$ is the area under the bell curve to the left of $z$.
- **Algebraic sigmoid:** $\displaystyle \frac12\left(\frac{z}{\sqrt{1 + z^2}} + 1\right)$
- **Rescaled tanh:** $\displaystyle \frac12\big(\tanh(z) + 1\big)$. Here $\tanh$ is the hyperbolic
  tangent, available as `np.tanh`, which goes from $-1$ to $1$; the rescaling moves it into $(0,1)$.

In [ ]:
from math import erf

def probit(z):
    return 0.5 * (1 + np.vectorize(erf)(z / np.sqrt(2)))    # standard normal CDF

def algebraic_sigmoid(z):
    return 0.5 * (z / np.sqrt(1 + z**2) + 1)

def tanh_rescaled(z):
    return 0.5 * (np.tanh(z) + 1)

zs = np.linspace(-6, 6, 400)
plt.figure(figsize=(7, 4))
for f, name in [(sigmoid, "logistic"), (probit, "probit"),
                (algebraic_sigmoid, "algebraic"), (tanh_rescaled, "tanh (rescaled)")]:
    plt.plot(zs, f(zs), linewidth=2, label=name)
plt.axhline(0.5, color="gray", linestyle=":")
plt.axvline(0, color="gray", linestyle=":")
plt.legend()
plt.title("Four ways to squash z into (0, 1)")
plt.xlabel("z")
plt.show()

All four pass through $(0, \frac12)$ and look similar. They differ in how **steep** they are in the
middle, and in how fast they approach 0 and 1 in the **tails**. The algebraic sigmoid, for example,
approaches them much more slowly than the others.

Does the choice matter for learning? We'll run that experiment in notebook 03a, after we've
implemented logistic regression ourselves. Until then we use the standard logistic function, and
section 4 shows one very good reason for that choice.

### Exercise 3.A: computing $h(x)$ by hand

A model with $n = 2$ features has $w = (0.5, -1)$ and $b = 1$.

1. For $x = (2, 1)$, compute $z = w^T x + b$ and then $h(x) = \sigma(z)$. Store them in `z_a`, `h_a`.
2. Do the same for $x = (0, 3)$, storing the results in `z_b`, `h_b`.

Do $z$ by hand first. For $\sigma$ you can use the `sigmoid` function above. Then check with NumPy:
`w @ x` computes $w^T x$.

In [ ]:
w = np.array([0.5, -1.0])
b = 1.0
# YOUR CODE HERE
z_a, h_a = None, None
z_b, h_b = None, None
check("3.A", [z_a, h_a, z_b, h_b])

### Exercise 3.B: your own sigmoid, and the symmetry property

1. Write `my_sigmoid(z)`, which should work on a whole NumPy array at once. Use `np.exp`.
2. Check the symmetry $\sigma(-z) = 1 - \sigma(z)$ numerically: for 1001 values of $z$ between $-10$
   and $10$, compute the **largest** absolute difference between the two sides. Store it in
   `max_symmetry_error`. It should be (practically) zero.

In [ ]:
def my_sigmoid(z):
    # YOUR CODE HERE
    return None

check_fn("3.B", my_sigmoid)

In [ ]:
zz = np.linspace(-10, 10, 1001)
# YOUR CODE HERE
max_symmetry_error = None
print("largest difference:", max_symmetry_error)
check("3.B2", max_symmetry_error, tol=1e-9)

### Checking a derivative numerically

We derived $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ with algebra. Can we check it **without** any
algebra, just with the computer? Yes, by going back to what a derivative *is*.

The derivative $f'(z)$ is the slope of the graph of $f$ at the point $z$. Take two points on the graph
that are very close to each other, and compute the slope of the line through them ("rise over run"):

$$f'(z) \approx \frac{f(z + \varepsilon) - f(z)}{\varepsilon} \qquad \text{(for a tiny } \varepsilon \text{, say } 10^{-5}\text{)}$$

This is exactly the definition of the derivative, just **without taking the limit** $\varepsilon \to 0$:
we stop at a small, fixed $\varepsilon$. It's called a **finite difference**.

A slightly better version takes one point on **each side** of $z$:

$$f'(z) \approx \frac{f(z + \varepsilon) - f(z - \varepsilon)}{2\varepsilon}$$

This is the **central difference**. It's more accurate because the errors on the two sides largely
cancel. Let's try both on a function whose derivative we know: $f(z) = z^2$, so $f'(3) = 6$.

In [ ]:
def f(z):
    return z ** 2

z0, eps = 3.0, 1e-5
one_sided = (f(z0 + eps) - f(z0)) / eps
central = (f(z0 + eps) - f(z0 - eps)) / (2 * eps)
print("true derivative:  6")
print("one-sided estimate:", one_sided)
print("central estimate:  ", central)

Both are very close to 6, and the central one is closer. This is a very useful habit:
**whenever you derive a derivative (or a gradient) by hand, check it numerically.** We'll use this
trick again when we implement logistic regression ourselves.

### Exercise 3.C: checking $\sigma'$ numerically

1. Write `numeric_derivative(f, z, eps=1e-5)`, which returns the central-difference estimate of the
   derivative of the function `f` at `z` (where `z` can be an array).
2. Use it to estimate $\sigma'(z)$ at the points in `z_points`, and store the result in `deriv_est`.
3. Compare it with the formula $\sigma(z)(1-\sigma(z))$ (printed for you).

In [ ]:
def numeric_derivative(f, z, eps=1e-5):
    # YOUR CODE HERE
    return None

z_points = np.linspace(-3, 3, 7)
# YOUR CODE HERE
deriv_est = None
print("numerical estimate:", deriv_est)
print("formula           :", sigmoid(z_points) * (1 - sigmoid(z_points)))
check("3.C", deriv_est, tol=1e-6)

### Exercise 3.D: log-odds

1. A model predicts $p = 0.8$. What are the **log-odds** $\log\frac{p}{1-p}$? Store them in
   `log_odds_08`. (`np.log` is the natural logarithm.)
2. The exam model $h(x) = \sigma(1.2x - 6)$ is given. How many hours must a student study for the
   model to predict a **90%** chance of passing? Store the answer in `hours_for_90`.
   *Hint:* don't search. Use the fact that the log-odds equal $1.2x - 6$, and solve for $x$.

In [ ]:
# YOUR CODE HERE
log_odds_08 = None
hours_for_90 = None
print(log_odds_08, hours_for_90)
check("3.D", [log_odds_08, hours_for_90])

### Exercise 3.E (a twist): design a model

Find $w$ and $b$ so that the one-feature model $h(x) = \sigma(wx + b)$ gives exactly
$h(2) = 0.1$ and $h(8) = 0.9$. Store them in `w_design` and `b_design`, then plot your curve.

*Hint:* turn each condition into a condition on the log-odds. You get two **linear** equations in $w$
and $b$.

In [ ]:
# YOUR CODE HERE
w_design = None
b_design = None
check("3.E", [w_design, b_design])

if w_design is not None and b_design is not None:
    xs = np.linspace(0, 10, 200)
    plt.plot(xs, sigmoid(w_design * xs + b_design), linewidth=2.5)
    plt.scatter([2, 8], [0.1, 0.9], color="red", zorder=3, label="required points")
    plt.legend()
    plt.title("Your designed model")
    plt.show()

## 4. The loss: how do we measure a bad fit?

In the previous unit we learned the recipe:

> choose a **loss function** $L(w, b)$ that measures how badly the hypothesis fits the data, and use
> **gradient descent** to find the $w, b$ that minimize it.

So which loss should we use? We compare two candidates. In both, $h_i = h(x_i)$ is the model's
predicted probability for example $i$, and $y_i \in \{0,1\}$ is its true label.

**Candidate 1: squared error ($L_2$)**, the loss from linear regression:
$$L_{2}(w, b) = \frac{1}{N} \sum_{i=1}^{N} \big(h_i - y_i\big)^2$$

**Candidate 2: log-loss** (also called **cross-entropy**):
$$L_{\log}(w, b) = -\frac{1}{N} \sum_{i=1}^{N} \Big[ y_i \log h_i + (1 - y_i) \log (1 - h_i) \Big]$$

The log-loss formula looks scary, but since $y_i$ is 0 or 1, only one of the two terms is ever
"on":

- if $y_i = 1$, the example contributes $-\log h_i$;
- if $y_i = 0$, the example contributes $-\log(1 - h_i)$.

In words: **minus the log of the probability the model gave to what actually happened.** (This is
where 0/1 labels make the formula clean.)

### 4.1 One example at a time

Take one example whose true label is $y = 1$. How much does each loss charge for each possible
prediction $h$?

In [ ]:
h = np.linspace(0.001, 0.999, 500)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, y_true in zip(axes, [1, 0]):
    if y_true == 1:
        ax.plot(h, (1 - h)**2, linewidth=2.5, label="squared error  (1 - h)^2")
        ax.plot(h, -np.log(h), linewidth=2.5, label="log-loss  -log(h)")
    else:
        ax.plot(h, h**2, linewidth=2.5, label="squared error  h^2")
        ax.plot(h, -np.log(1 - h), linewidth=2.5, label="log-loss  -log(1 - h)")
    ax.set_ylim(0, 5)
    ax.set_xlabel("predicted probability h")
    ax.set_ylabel("loss for this example")
    ax.set_title(f"true label y = {y_true}")
    ax.legend()
plt.show()

for h_val in [0.9, 0.5, 0.1, 0.01, 0.001]:
    print(f"y = 1, prediction h = {h_val:<6}  squared error = {(1 - h_val)**2:.3f}   log-loss = {-np.log(h_val):.3f}")

Look at the left panel as the prediction $h$ moves toward 0, i.e. the model becomes **confidently
wrong**:

- **Squared error** never charges more than 1. Being 99.9% sure of the wrong answer costs almost
  the same as being 90% sure of it.
- **Log-loss** grows without limit ($-\log h \to \infty$ as $h \to 0$). Confidently wrong answers
  are punished heavily, which is exactly what we want from a model that outputs probabilities.

That's a hint. The real difference, though, shows up when we try to **minimize** the losses with
gradient descent.

### 4.2 The loss landscape on the exam data

Let's compute both losses on the exam data for many values of $w$ and $b$, and look at the landscape
that gradient descent has to walk on.

In [ ]:
def squared_error_loss(w, b):
    h = sigmoid(w * hours + b)
    return np.mean((h - passed) ** 2)

def log_loss_wb(w, b):
    h = np.clip(sigmoid(w * hours + b), 1e-12, 1 - 1e-12)   # avoid log(0)
    return -np.mean(passed * np.log(h) + (1 - passed) * np.log(1 - h))

# first, a 1D slice: fix b = -3 and vary w
ws = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(ws, [squared_error_loss(w, -3) for w in ws], linewidth=2.5)
axes[0].set_title("squared error, b = -3")
axes[1].plot(ws, [log_loss_wb(w, -3) for w in ws], linewidth=2.5, color="C1")
axes[1].set_title("log-loss, b = -3")
for ax in axes:
    ax.set_xlabel("w")
plt.show()

Two very different shapes:

- **Squared error** has a long, completely **flat plateau** on the left, and on the right it
  bends the wrong way (it isn't bowl-shaped). A function like this is called **non-convex**.
- **Log-loss** is a single smooth **bowl**: **convex**. From anywhere, "downhill" leads to the
  bottom.

**Try it yourself, before reading on:** we run gradient descent starting somewhere on the flat part of
the squared-error curve, say $w = -2$. What will happen?

In [ ]:
def run_gd(loss, w, b, eta, n_steps):
    # Plain gradient descent on the exam data. Returns the path of (w, b) values.
    path = [(w, b)]
    for _ in range(n_steps):
        h = sigmoid(w * hours + b)
        if loss == "squared":
            g = 2 * (h - passed) * h * (1 - h)     # d(loss)/dz for squared error (see 4.3)
        else:
            g = h - passed                          # d(loss)/dz for log-loss (see 4.3)
        w = w - eta * np.mean(g * hours)
        b = b - eta * np.mean(g)
        path.append((w, b))
    return np.array(path)

start = (-2.0, 0.0)
path_sq = run_gd("squared", *start, eta=0.5, n_steps=5000)
path_log = run_gd("log", *start, eta=0.1, n_steps=5000)

for name, path in [("squared error", path_sq), ("log-loss", path_log)]:
    w_end, b_end = path[-1]
    acc = np.mean((sigmoid(w_end * hours + b_end) > 0.5) == passed)
    print(f"{name:14s}: ended at w = {w_end:6.3f}, b = {b_end:6.3f}   training accuracy = {acc:.3f}")

In [ ]:
W, B = np.meshgrid(np.linspace(-4, 2.5, 150), np.linspace(-9, 3, 150))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, lossfn, path, title in [(axes[0], squared_error_loss, path_sq, "squared error"),
                                (axes[1], log_loss_wb, path_log, "log-loss")]:
    Z = np.vectorize(lossfn)(W, B)
    cs = ax.contourf(W, B, Z, levels=30, cmap="viridis")
    fig.colorbar(cs, ax=ax)
    ax.plot(path[:, 0], path[:, 1], "w.-", markersize=2, linewidth=1)
    ax.plot(*path[0], "wo", markersize=8, label="start")
    ax.plot(*path[-1], "r*", markersize=15, label="end")
    ax.set_xlabel("w"); ax.set_ylabel("b"); ax.set_title(f"{title}: gradient descent path")
    ax.legend(loc="lower left")
plt.show()

**Squared error got stuck.** It ended far from the good region, with a model that predicts
"fail" for nearly everyone (look at its training accuracy). It isn't a local minimum in the usual
sense: the landscape there is just **so flat that the gradient is almost zero**, so gradient descent
barely moves. A larger learning rate doesn't rescue it (try `eta=5` in the cell above).

**Log-loss found the bottom of the bowl**, from exactly the same starting point.

To see *why* the squared-error gradient vanishes, let's derive the gradients.

### 4.3 The gradient of the log-loss, step by step

Take **one example** $(x, y)$ and write its log-loss as a chain of three simple steps:

$$z = w^T x + b \quad\longrightarrow\quad h = \sigma(z) \quad\longrightarrow\quad
\ell = -\big[y \log h + (1-y)\log(1-h)\big]$$

We use the chain rule, $\dfrac{\partial \ell}{\partial w_j} = \dfrac{\partial \ell}{\partial h}\cdot
\dfrac{\partial h}{\partial z} \cdot \dfrac{\partial z}{\partial w_j}$, one factor at a time.

**Step 1**, the loss with respect to $h$:
$$\frac{\partial \ell}{\partial h} = -\frac{y}{h} + \frac{1-y}{1-h}
= \frac{-y(1-h) + (1-y)h}{h(1-h)} = \frac{h - y}{h(1-h)}$$

**Step 2**, $h$ with respect to $z$ (the neat derivative from 3.2):
$$\frac{\partial h}{\partial z} = h(1-h)$$

**Step 3**, $z$ with respect to the parameters:
$$\frac{\partial z}{\partial w_j} = x_j, \qquad \frac{\partial z}{\partial b} = 1$$

**Multiply.** The $h(1-h)$ cancels, which is the magic moment:
$$\boxed{\;\frac{\partial \ell}{\partial w_j} = (h - y)\, x_j, \qquad \frac{\partial \ell}{\partial b} = h - y\;}$$

For the whole dataset we average over the examples:
$$\frac{\partial L}{\partial w_j} = \frac{1}{N}\sum_{i=1}^{N} (h_i - y_i)\, x_{i,j}, \qquad
\frac{\partial L}{\partial b} = \frac{1}{N}\sum_{i=1}^{N} (h_i - y_i)$$

or, in vector form, $\nabla_w L = \frac{1}{N}\sum_i (h_i - y_i)\, x_i$.

**Compare with linear regression.** For squared error with $h = w^T x + b$, the gradient was
$\frac{2}{N}\sum_i (h_i - y_i)\, x_i$: **the same form**, "error times input". The only difference is
that $h$ now goes through $\sigma$. This clean formula is the main reason the logistic function and
log-loss are always used together.

**And the squared error?** For one example, $\ell = (h - y)^2$, so
$\frac{\partial \ell}{\partial h} = 2(h - y)$. Multiplying by $\frac{\partial h}{\partial z} = h(1-h)$,
this time **nothing cancels**:

$$\frac{\partial \ell}{\partial w_j} = 2(h - y)\;\underbrace{h(1-h)}_{\text{tiny when } h \approx 0 \text{ or } 1}\; x_j$$

When the model is **confidently wrong** ($y = 1$ but $h \approx 0$), the factor $h(1-h)$ is almost
zero. The gradient almost vanishes, and gradient descent can't fix the mistake: the plateau we got
stuck on. With log-loss the factor cancelled, so a confidently wrong model gets a **strong** push
($h - y \approx -1$).

### 4.4 Optional: where does log-loss come from?

*This part is optional; the rest of the unit doesn't depend on it.*

**A riddle.** In country X, the weather in the last ten days was:

<p style="font-size: 1.3em; letter-spacing: 0.3em; text-align: center"><b>S R S S R R S R S S</b></p>

where S means a sunny day and R a rainy day. **How would you estimate the probability of a rainy day
in country X?**

**Try it yourself, before reading on:** what's your estimate, and *why*?

Almost everyone answers $\frac{4}{10} = 0.4$: 4 rainy days out of 10. It feels obvious. But can
we **justify** it? Why not 0.3, or 0.5?

Here's a principled way to think about it. Suppose the chance of rain on any day is some number $p$,
independently of other days. Then the probability of seeing **exactly this sequence** of ten days is

$$(1-p)\cdot p\cdot(1-p)\cdot(1-p)\cdot p\cdot p\cdot(1-p)\cdot p\cdot(1-p)\cdot(1-p) \;=\; p^4\,(1-p)^6$$

(a factor $p$ for every R and a factor $1-p$ for every S). This number depends on $p$, and it's
called the **likelihood** of $p$. A sensible choice is the $p$ under which what we actually saw is
**most likely**. Let's plot it.

In [ ]:
days = "S R S S R R S R S S".split()
n_rain, n_sun = days.count("R"), days.count("S")

p = np.linspace(0.001, 0.999, 999)      # steps of 0.001
likelihood = p ** n_rain * (1 - p) ** n_sun
log_likelihood = n_rain * np.log(p) + n_sun * np.log(1 - p)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(p, likelihood, linewidth=2.5)
axes[0].set_title("likelihood  p^4 (1-p)^6")
axes[1].plot(p, log_likelihood, linewidth=2.5, color="C1")
axes[1].set_ylim(-20, 0)
axes[1].set_title("log-likelihood  4 log p + 6 log(1-p)")
for ax in axes:
    ax.axvline(0.4, color="gray", linestyle="--")
    ax.set_xlabel("p (chance of rain)")
plt.show()

print("p with the highest likelihood:", p[np.argmax(likelihood)].round(3))

The likelihood peaks at $p = 0.4$: the intuitive answer is exactly the **maximum-likelihood**
answer. Notice two things:

- The **log**-likelihood (right) peaks at the **same** $p$. That's because $\log$ is increasing: making
  a positive number bigger always makes its log bigger. Logs are easier to work with, because they
  turn products into sums.
- We can find the peak with calculus: the derivative of $4\log p + 6\log(1-p)$ is
  $\frac{4}{p} - \frac{6}{1-p}$. Setting it to zero gives $4(1-p) = 6p$, so $p = 0.4$.

**From the riddle to logistic regression.** Write each day as a label: $y_i = 1$ for rain, $y_i = 0$
for sun. The probability of day $i$'s weather is $p$ if $y_i = 1$ and $1-p$ if $y_i = 0$. A compact
way to write both cases is $p^{\,y_i}(1-p)^{1-y_i}$ (check it by plugging in $y_i = 1$ and $y_i = 0$).

In the riddle, $p$ was **the same every day**. But in real life the chance of rain depends on the
conditions: today's pressure, humidity, and so on. That's exactly what logistic regression models: day
$i$ with features $x_i$ gets its **own** probability $h_i = \sigma(w^T x_i + b)$. The likelihood of all
the data becomes

$$\prod_{i=1}^N h_i^{\,y_i}(1-h_i)^{1-y_i}$$

and, just as in the riddle, we take the log to turn the product into a sum:

$$\sum_{i=1}^N \big[y_i \log h_i + (1-y_i)\log(1-h_i)\big]$$

**Maximizing** this is the same as **minimizing minus it** (divided by $N$), and that is exactly the
log-loss! So:

> *Minimizing log-loss = choosing the $w, b$ under which the data we observed is most likely.*

This principle is called **maximum likelihood**. It's the same idea that makes "4 out of 10" the
natural answer to the riddle.

**Optional exercise 4.E.** Generalize the riddle: in $N$ days there were $k$ rainy days. Show that
the $p$ with the highest likelihood is $p = \frac{k}{N}$.

**Your answer:** *(write here)*

*A bonus connection:* logistic regression with **no features** is $h = \sigma(b)$, the same probability
for every example, exactly like the riddle. Training it with log-loss gives $\sigma(b) = \frac{k}{N}$.
It "learns" the fraction of positives.

### Exercise 4.A: losses by hand

Three examples have these true labels and predicted probabilities:

| example | $y$ | $h$ |
|---|---|---|
| 1 | 1 | 0.9 |
| 2 | 0 | 0.2 |
| 3 | 1 | 0.3 |

Compute by hand (a calculator for $\log$ is fine) the **log-loss** and the **squared error** of the
three examples together. Store them in `logloss_by_hand` and `sqerr_by_hand`. Which example
contributes the most to the log-loss, and why?

In [ ]:
# YOUR CODE HERE
logloss_by_hand = None
sqerr_by_hand = None
print(logloss_by_hand, sqerr_by_hand)
check("4.A", [logloss_by_hand, sqerr_by_hand])

**Your answer:** *(write here)*

### Exercise 4.B: loss functions in code

Write two functions that take an array of labels `y` and an array of predicted probabilities `h` of
the same length, and return a **single number**:

- `log_loss(y, h)`: the average log-loss;
- `squared_error(y, h)`: the average squared error.

Use NumPy operations on the whole arrays, with no loops.

In [ ]:
def log_loss(y, h):
    # YOUR CODE HERE
    return None

def squared_error(y, h):
    # YOUR CODE HERE
    return None

check_fn("4.B", lambda y, h: [log_loss(y, h), squared_error(y, h)]
         if log_loss(y, h) is not None and squared_error(y, h) is not None else None)

### Exercise 4.C: the gradient for one example

Write `gradient_one(x, y, w, b)`, which returns the gradient of the log-loss of a **single** example,
using the boxed formula from 4.3. It should return a pair `(grad_w, grad_b)`, where `grad_w` is an
array the same length as `x`.

Then use `numeric_derivative`-style finite differences to confirm it: change $w_1$ by a tiny amount
and see how the loss changes. (The second cell does this for you once your function works.)

In [ ]:
def gradient_one(x, y, w, b):
    # YOUR CODE HERE
    return None, None

x_ex, y_ex = np.array([2.0, 1.0]), 0
w_ex, b_ex = np.array([0.5, -1.0]), 0.5
gw, gb = gradient_one(x_ex, y_ex, w_ex, b_ex)
print("grad_w =", gw, "  grad_b =", gb)
check("4.C", None if gw is None else [gw[0], gw[1], gb])

In [ ]:
# finite-difference check of the w_1 component
def loss_one(w, b):
    h = sigmoid(w @ x_ex + b)
    return -(y_ex * np.log(h) + (1 - y_ex) * np.log(1 - h))

eps = 1e-5
e1 = np.array([1.0, 0.0])
numeric = (loss_one(w_ex + eps * e1, b_ex) - loss_one(w_ex - eps * e1, b_ex)) / (2 * eps)
print("numerical  d(loss)/d(w_1):", numeric)
print("your formula             :", None if gw is None else gw[0])

### Exercise 4.D (a twist): why squared error gets stuck, in numbers

A positive example ($y = 1$) with $x = 1$ gets score $z = -7$, so the model is confidently wrong.

1. Compute $h = \sigma(-7)$.
2. Compute the gradient with respect to $z$ under both losses:
   - log-loss: $\;h - y$
   - squared error: $\;2(h - y)\,h(1-h)$

   Store them in `g_log` and `g_sq`.
3. How many times stronger is the log-loss push? Print `g_log / g_sq`.

In [ ]:
# YOUR CODE HERE
g_log = None
g_sq = None
print("log-loss gradient:     ", g_log)
print("squared-error gradient:", g_sq)
if g_log is not None and g_sq is not None:
    print("ratio:", g_log / g_sq)
check("4.D", [g_log, g_sq], tol=1e-5)

**Your answer:** *(write here)*

## 5. Putting it all together

| | Linear regression | Perceptron | **Logistic regression** |
|---|---|---|---|
| **Target function** | $x \mapsto$ a number | $x \mapsto$ a class | $x \mapsto P(y = 1 \mid x)$ |
| **Data** | $(x_i, y_i)$, $y_i \in \mathbb{R}$ | $(x_i, y_i)$, $y_i \in \{-1, +1\}$ | $(x_i, y_i)$, $y_i \in \{0, 1\}$ |
| **Hypothesis class** | $h(x) = w^T x + b$ | $h(x) = \text{sign}(w^T x + b)$ | $h(x) = \sigma(w^T x + b)$ |
| **Loss** | squared error | (perceptron rule) | **log-loss** |
| **Learning algorithm** | gradient descent (or a formula) | perceptron updates | **gradient descent** |
| **Gradient** | $\frac{2}{N}\sum_i (h_i - y_i)\,x_i$ | n/a | $\frac{1}{N}\sum_i (h_i - y_i)\,x_i$ |

**Logistic regression, in one sentence:** *a linear score $w^T x + b$, turned into a probability by
the logistic function, trained by minimizing log-loss with gradient descent.*

Let's watch the full recipe work on the exam data. We start from $w = 0$, $b = 0$ (the model predicts
0.5 for everyone) and run gradient descent on the log-loss.

In [ ]:
w_fit, b_fit = 0.0, 0.0
eta = 0.1
loss_history = []
for step in range(5000):
    h = sigmoid(w_fit * hours + b_fit)
    loss_history.append(log_loss_wb(w_fit, b_fit))
    w_fit -= eta * np.mean((h - passed) * hours)     # the gradient from 4.3
    b_fit -= eta * np.mean(h - passed)

print(f"learned: w = {w_fit:.3f}, b = {b_fit:.3f}")
print(f"log-loss went from {loss_history[0]:.3f} to {loss_history[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(loss_history)
axes[0].set_xlabel("gradient descent step"); axes[0].set_ylabel("log-loss")
axes[0].set_title("training: log-loss goes down")
xs = np.linspace(0, 10, 200)
axes[1].scatter(hours, passed, c=passed, cmap="coolwarm", edgecolor="k", zorder=3)
axes[1].plot(xs, sigmoid(w_fit * xs + b_fit), "g-", linewidth=2.5, label="learned h(x)")
axes[1].plot(xs, 1 / (1 + np.exp(-1.2 * (xs - 5))), "k:", label="true probability (hidden)")
axes[1].set_xlabel("hours studied"); axes[1].set_ylabel("P(pass)")
axes[1].legend(loc="lower right")
axes[1].set_title("the learned model")
plt.show()

The learned curve is close to the true probability curve that generated the data, which the
algorithm never saw: it only saw the 0/1 outcomes. That's what logistic regression does, recovering a
probability from yes/no data.

(The fit isn't perfect, because 40 students is a small sample. With more data the two curves would
get closer.)

### Exercise 5.A: the framework for a new problem

A school wants to predict, from a student's record, whether they'll **drop a course** before the end of
the semester. Fill in the framework, as in the table above:

- target function, data, $n$ (suggest features), hypothesis class, loss, learning algorithm;
- what would an output of $h(x) = 0.15$ mean, in words?

**Your answer:** *(write here)*

### Exercise 5.B: using the trained model

Use `w_fit` and `b_fit` from the training above.

1. What is the predicted probability of passing for a student who studies **5 hours**? Store it in
   `p_5h`.
2. How many hours does a student need for a **75%** predicted chance? Store the answer in `hours_75`.
   (Log-odds again.)

In [ ]:
# YOUR CODE HERE
p_5h = None
hours_75 = None
print(p_5h, hours_75)
check("5.B", [p_5h, hours_75], tol=1e-2)

### Exercise 5.C: predict, then run

Answer each question **before** running anything, then check by changing the code of the training
cell (or of `run_gd` in 4.2) and running it.

1. In the training cell, what happens if the learning rate is $\eta = 0.001$ instead of $0.1$?
2. In 4.2, gradient descent on **log-loss** started from $(w, b) = (-2, 0)$. What would happen if it
   started from $(w, b) = (5, 5)$?
3. What happens to the learned $w$ if we flip all the labels (`1 - passed`) and train again?

**Your answer:** *(write here)*

## Recap

- **Probability estimation** is a third kind of question, besides regression and classification: we
  want $P(y = 1 \mid x)$, a number in $[0, 1]$, even though the data only has 0/1 labels.
- From now on, **labels are $y \in \{0,1\}$** (the perceptron used $\pm 1$).
- **Hypothesis class:** $h(x) = \sigma(w^T x + b)$ with the **logistic function**
  $\sigma(z) = \frac{1}{1+e^{-z}}$.
  - Its outputs are always in $(0, 1)$, with $\sigma(0) = \frac12$.
  - It is symmetric: $\sigma(-z) = 1 - \sigma(z)$.
  - Its derivative is $\sigma' = \sigma(1-\sigma)$.
  - Close up it looks linear; zoomed out it looks like the perceptron's step.
  - The model is **linear in the log-odds**: $\log\frac{h}{1-h} = w^T x + b$.
- Other sigmoids exist (probit, algebraic, rescaled tanh). We'll test them in 03a.
- **Loss: log-loss**, "minus the log of the probability given to what actually happened".
  - Squared error with a sigmoid is non-convex, with flat plateaus where gradient descent gets stuck.
  - Log-loss is convex.
- **Gradient:** $\frac{\partial L}{\partial w} = \frac1N\sum_i (h_i - y_i)\,x_i$ and
  $\frac{\partial L}{\partial b} = \frac1N\sum_i (h_i - y_i)$. It has the same "error × input" form
  as linear regression.
- **Next (01b):** logistic regression with two features, decision boundaries, turning probabilities
  into classes, and measuring how good a classifier is.

## References and further reading

**Interactive explanations (highly recommended)**

- **[Logistic Regression](https://mlu-explain.github.io/logistic-regression/)**, by Erin Bugbee and
  Jared Wilber (MLU-Explain). A visual, interactive walk through this notebook's ideas: the sigmoid,
  probabilities, log-odds and log-loss.

**Short readings**

- Google Machine Learning Crash Course, *Logistic regression* module:
  - [Calculating a probability](https://developers.google.com/machine-learning/crash-course/logistic-regression/sigmoid-function):
    the sigmoid function and log-odds (sections 3.1 and 3.2 here);
  - [Loss and regularization](https://developers.google.com/machine-learning/crash-course/logistic-regression/loss-regularization):
    log-loss (section 4 here). The regularization part is a preview of notebook 03b.

**Videos**

- StatQuest with Josh Starmer, the
  [Logistic Regression playlist](https://www.youtube.com/playlist?list=PLblh5JKOoLUKxzEP5HA2d-Li7IJkHfXSe).
  Short, friendly videos, including one on maximum likelihood (our optional section 4.4).

**Documentation (for looking things up)**

- NumPy: [`np.exp`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html) and
  [`np.log`](https://numpy.org/doc/stable/reference/generated/numpy.log.html).
- scikit-learn: [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html),
  used in section 1.